# Broad Value Screen v3 — S&P 1500, price-first

Recovered 2026-08-12 from the Colab Drive archive (see `wiki/colab-archive-audit.md`).

**⚠️ RENAMED ON RECOVERY.** In Drive this file was called `cluster_hunter.ipynb`, which COLLIDES with the
repo's existing `tools/cluster_hunter.ipynb` — and they are **not two versions of one tool, they are two
different tools**: the repo's reads OpenInsider cluster-buy feeds; this one runs a two-stage value screen
over the S&P 1500. Same name, different job. Renamed so neither overwrites the other.

**Design:** Stage 1 = ONE batched price pull of the S&P 1500, keep only names under $60 (fast).
Stage 2 = fundamentals on the survivors ONLY, checkpointed so it is safe to stop and re-run.
Final gates: px<50 · profitable · FCF yield >=5% · revenue growth >=0 · fwdEPS>trailEPS · D/E<1.5.

**Its 8/? archived run:** 1,506 names -> 625 under $60 -> **89 survivors**. Requires yfinance, so it is a
**Colab run** — this container gets HTTP 429 from Yahoo (2026-08-12).


In [ ]:
# BROAD VALUE SCREEN v3 — price-first for speed. Complete cell, run as-is.
# Stage 1: ONE batch price pull of the S&P 1500 → keep only <$60 (fast, ~2 min).
# Stage 2: fundamentals ONLY on those survivors (checkpointed — safe to stop/rerun).
# Final gates: px<50, profitable, FCF yield>=5%, revG>=0, fwdEPS>trailEPS, D/E<1.5.
import yfinance as yf, pandas as pd, requests, io, time, os

H = {"User-Agent": "Mozilla/5.0 (research; contact 7bm4q6x5sm@privaterelay.appleid.com)"}
tables = {
 "500": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
 "400": "https://en.wikipedia.org/wiki/List_of_S%26P_400_companies",
 "600": "https://en.wikipedia.org/wiki/List_of_S%26P_600_companies",
}
tickers = set()
for k, url in tables.items():
    try:
        html = requests.get(url, headers=H, timeout=30).text
        for tbl in pd.read_html(io.StringIO(html)):
            col = next((c for c in tbl.columns if str(c).lower() in ("symbol", "ticker", "ticker symbol")), None)
            if col is not None and len(tbl) > 50:
                tickers |= set(tbl[col].astype(str).str.replace(".", "-", regex=False))
                break
        print(f"S&P {k}: total so far {len(tickers)}")
    except Exception as e:
        print(f"S&P {k} fetch fail: {str(e)[:60]}")
tickers = sorted(tickers)
assert len(tickers) > 1000, "constituent fetch failed — stop and tell Claude"

# ---- STAGE 1: price cut <$60 ----
px_parts = []
for i in range(0, len(tickers), 250):
    try:
        d = yf.download(tickers[i:i+250], period="1d", progress=False, auto_adjust=True)["Close"]
        px_parts.append(d.iloc[-1] if hasattr(d, "iloc") else d)
    except Exception as e:
        print(f"chunk {i}: {str(e)[:40]}")
    time.sleep(1)
px = pd.concat(px_parts).dropna()
survivors = px[(px > 1) & (px < 60)].index.tolist()
print(f"\nStage 1 done: {len(survivors)} names under $60 (of {len(tickers)}) — pulling fundamentals on these only")

# ---- STAGE 2: fundamentals on survivors, checkpointed ----
CKPT = "screen_ckpt.csv"
done = set(pd.read_csv(CKPT).tkr) if os.path.exists(CKPT) else set()
if done: print(f"resuming: {len(done)} already done")
rows = []
def save():
    parts = [pd.DataFrame(rows)]
    if os.path.exists(CKPT): parts.insert(0, pd.read_csv(CKPT))
    pd.concat(parts).drop_duplicates("tkr").to_csv(CKPT, index=False)
for n, t in enumerate(survivors):
    if t in done: continue
    try:
        i = yf.Ticker(t).info
        mc = i.get("marketCap"); fcf = i.get("freeCashflow")
        te, fe = i.get("trailingEps"), i.get("forwardEps")
        rg = i.get("revenueGrowth"); de = i.get("debtToEquity")
        rows.append({"tkr": t, "px": round(float(px[t]), 2), "sector": i.get("sector", "?"),
                     "industry": str(i.get("industry", "?"))[:28],
                     "mcap_B": round(mc/1e9, 2) if mc else None,
                     "fcf_yield%": round(fcf/mc*100, 1) if fcf and mc else None,
                     "trailEPS": te, "fwdEPS": fe,
                     "fwdPE": round(float(px[t])/fe, 1) if fe and fe > 0 else None,
                     "revG%": round(rg*100, 1) if rg is not None else None,
                     "D/E": round(de/100, 2) if de is not None else None})
    except Exception:
        rows.append({"tkr": t})
    if len(rows) % 50 == 0:
        save(); print(f"  ...{n+1}/{len(survivors)}")
save()
df = pd.read_csv(CKPT)

# ---- FINAL GATES ----
g = df.dropna(subset=["fcf_yield%", "trailEPS", "fwdEPS"])
g = g[(g.px < 50) & (g.trailEPS > 0) & (g["fcf_yield%"] >= 5) & (g.fwdEPS > g.trailEPS)
      & (g["revG%"].fillna(-1) >= 0) & (g["D/E"].fillna(99) < 1.5)]
g = g.sort_values("fcf_yield%", ascending=False)
print(f"\n=== SURVIVORS: {len(g)} ===")
print(g[["tkr","px","sector","industry","mcap_B","fcf_yield%","fwdPE","revG%","D/E"]].to_string(index=False))


In [ ]:
# INSIDER PULL — broad-screen shortlist, 90 days. Complete cell, run as-is, paste output back.
import requests, time, re
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import date, timedelta

H = {"User-Agent": "Jake research 7bm4q6x5sm@privaterelay.appleid.com"}
TICKERS = ["PFE", "PYPL", "LKQ", "ABM", "DLB", "GNTX", "WS", "AR", "MAT", "ADNT", "FLO", "AMPH"]
SINCE = (date.today() - timedelta(days=90)).isoformat()

cm = requests.get("https://www.sec.gov/files/company_tickers.json", headers=H, timeout=30).json()
cik = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in cm.values()}

def txt(el, path):
    e = el.find(path)
    return e.text.strip() if e is not None and e.text else None

rows = []
for t in TICKERS:
    c = cik.get(t)
    if not c:
        print(f"{t}: no CIK"); continue
    sub = requests.get(f"https://data.sec.gov/submissions/CIK{c}.json", headers=H, timeout=30).json()
    time.sleep(0.12)
    rec = sub["filings"]["recent"]
    f4s = [(rec["accessionNumber"][i], rec["filingDate"][i], rec["primaryDocument"][i])
           for i in range(len(rec["form"]))
           if rec["form"][i] == "4" and rec["filingDate"][i] >= SINCE]
    for acc, fdate, pdoc in f4s:
        url = f"https://www.sec.gov/Archives/edgar/data/{int(c)}/{acc.replace('-','')}/{pdoc.split('/')[-1]}"
        try:
            x = requests.get(url, headers=H, timeout=30).text
            time.sleep(0.12)
            root = ET.fromstring(re.sub(r"<\?xml[^>]*\?>", "", x, count=1))
        except Exception as e:
            print(f"  {t} {acc}: parse fail {str(e)[:35]}"); continue
        owner = txt(root, ".//reportingOwner/reportingOwnerId/rptOwnerName") or "?"
        rel = root.find(".//reportingOwner/reportingOwnerRelationship")
        title = (txt(rel, "officerTitle") or
                 ("Director" if txt(rel, "isDirector") in ("1", "true") else "") or
                 ("10%+ owner" if txt(rel, "isTenPercentOwner") in ("1", "true") else "?"))
        plan = txt(root, ".//aff10b5One") in ("1", "true")
        for tr in root.findall(".//nonDerivativeTransaction"):
            code = txt(tr, "transactionCoding/transactionCode")
            sh = txt(tr, "transactionAmounts/transactionShares/value")
            pr = txt(tr, "transactionAmounts/transactionPricePerShare/value")
            if not (code and sh):
                continue
            rows.append({"tkr": t, "filed": fdate, "owner": owner[:24], "title": title[:20],
                         "code": code, "value_$": round(float(sh) * float(pr or 0)),
                         "10b5-1": "YES" if plan else ""})
    print(f"{t}: {len(f4s)} Form 4s")

df = pd.DataFrame(rows)
if len(df):
    print("\n=== SUMMARY (discretionary only) ===")
    disc = df[df["10b5-1"] != "YES"]
    for t in TICKERS:
        d = disc[disc.tkr == t]
        b = d[d.code == "P"]; s = d[d.code == "S"]
        bs, ss, nb = b["value_$"].sum(), s["value_$"].sum(), b.owner.nunique()
        tag = " <-- CLUSTER" if nb >= 3 else ""
        print(f"{t}: buys ${bs:,} ({nb} buyers{tag}) | disc sells ${ss:,}")
    print("\n=== BUY DETAIL ===")
    print(df[df.code == "P"].sort_values(["tkr", "filed"]).to_string(index=False))
else:
    print("no transactions in window")


In [ ]:
# OIL OPTIONS SCAN — USO + BNO, both wings + spread economics. Complete cell, run as-is.
import yfinance as yf, pandas as pd
from datetime import date

for TICK in ["USO", "BNO"]:
    t = yf.Ticker(TICK)
    spot = t.history(period="1d")["Close"].iloc[-1]
    print(f"\n{'='*70}\n{TICK} spot ${spot:.2f}\n{'='*70}")
    for exp in t.options[:8]:
        dte = (date.fromisoformat(exp) - date.today()).days
        if dte < 10 or dte > 100: continue
        ch = t.option_chain(exp)
        puts, calls = ch.puts, ch.calls
        # window: puts 80-100% of spot, calls 100-120%
        p = puts[(puts.strike >= spot*0.80) & (puts.strike <= spot*1.00)].copy()
        c = calls[(calls.strike >= spot*1.00) & (calls.strike <= spot*1.20)].copy()
        if len(p) < 2 or len(c) < 2: continue
        p["m%"] = (p.strike/spot*100).round(1); c["m%"] = (c.strike/spot*100).round(1)
        print(f"\n--- {exp} ({dte}d) ---")
        print("PUTS:")
        print(p[["strike","m%","bid","ask","impliedVolatility","volume","openInterest"]]
              .assign(impliedVolatility=lambda d:(d.impliedVolatility*100).round(1))
              .to_string(index=False))
        print("CALLS:")
        print(c[["strike","m%","bid","ask","impliedVolatility","volume","openInterest"]]
              .assign(impliedVolatility=lambda d:(d.impliedVolatility*100).round(1))
              .to_string(index=False))
        # skew: ~90% put IV minus ~110% call IV
        try:
            p90 = p.iloc[(p.strike - spot*0.90).abs().argsort()].iloc[0]
            c110 = c.iloc[(c.strike - spot*1.10).abs().argsort()].iloc[0]
            skew = (p90.impliedVolatility - c110.impliedVolatility) * 100
            print(f"SKEW (90% put IV − 110% call IV): {skew:+.1f} pts "
                  f"({'PUTS richer — crowd is short, selling puts/put spreads pays' if skew > 3 else 'CALLS richer — war premium alive, selling call spreads pays' if skew < -3 else 'balanced'})")
        except Exception: pass
        # spread economics
        try:
            pl = p.iloc[(p.strike - spot*0.93).abs().argsort()].iloc[0]   # long ~93%
            ps = p.iloc[(p.strike - spot*0.86).abs().argsort()].iloc[0]   # short ~86%
            if pl.strike > ps.strike:
                cost = pl.ask - ps.bid; width = pl.strike - ps.strike
                print(f"PUT DEBIT SPREAD {pl.strike}/{ps.strike}: cost ~${cost:.2f}, "
                      f"max ${width - cost:.2f} ({(width-cost)/cost*100:.0f}% max gain), "
                      f"breakeven ${pl.strike - cost:.2f}")
            cs = c.iloc[(c.strike - spot*1.08).abs().argsort()].iloc[0]   # short ~108%
            cl = c.iloc[(c.strike - spot*1.15).abs().argsort()].iloc[0]   # long ~115%
            if cl.strike > cs.strike:
                cred = cs.bid - cl.ask; width = cl.strike - cs.strike
                if cred > 0:
                    print(f"CALL CREDIT SPREAD {cs.strike}/{cl.strike}: credit ~${cred:.2f}, "
                          f"max loss ${width - cred:.2f}, return-on-risk {cred/(width-cred)*100:.0f}%, "
                          f"safe below ${cs.strike}")
        except Exception: pass


In [ ]:
# CSP + CHEAP-VOL SCAN — vetted names, ~2wk expiry, collateral-capped. Complete cell, run as-is.
import yfinance as yf, pandas as pd, numpy as np
from datetime import date

NAMES = ["ADT","GPK","AMPH","MAT","FLO","LKQ","ABM","USAC","VZ","PYPL","PFE","GEHC","ADSK","UPWK","BLKB"]
MAX_COLLATERAL = 2500        # <- your cap: strike*100 must be under this
TARGET_DTE = (7, 21)         # ~2-week window

rows = []
for tk in NAMES:
    try:
        t = yf.Ticker(tk)
        h = t.history(period="3mo")["Close"]
        spot = h.iloc[-1]
        rv20 = np.log(h).diff().dropna()[-20:].std() * np.sqrt(252) * 100   # realized vol, 20d
        exp = next((e for e in t.options
                    if TARGET_DTE[0] <= (date.fromisoformat(e) - date.today()).days <= TARGET_DTE[1]), None)
        if not exp: continue
        dte = (date.fromisoformat(exp) - date.today()).days
        puts = t.option_chain(exp).puts
        p = puts[(puts.strike >= spot*0.85) & (puts.strike <= spot*1.00) & (puts.bid > 0)]
        for _, r in p.iterrows():
            coll = r.strike * 100
            if coll > MAX_COLLATERAL and tk not in ("GEHC","ADSK"):  # show the big two anyway for reference
                continue
            iv = r.impliedVolatility * 100
            rows.append({"tkr": tk, "spot": round(spot,2), "exp": exp, "dte": dte,
                         "strike": r.strike, "OTM%": round((1 - r.strike/spot)*100,1),
                         "bid": r.bid, "collateral_$": int(coll),
                         "yield%": round(r.bid/r.strike*100, 2),
                         "annualized%": round(r.bid/r.strike*365/dte*100, 1),
                         "IV": round(iv,1), "RV20": round(rv20,1),
                         "IV/RV": round(iv/rv20, 2),
                         "breakeven": round(r.strike - r.bid, 2),
                         "OI": int(r.openInterest or 0)})
    except Exception as e:
        print(f"{tk}: {str(e)[:40]}")

df = pd.DataFrame(rows)
if len(df):
    df = df.sort_values("annualized%", ascending=False)
    print("=== CSP CANDIDATES (~2wk, collateral-capped) — sorted by annualized yield ===")
    print(df.to_string(index=False))
    print("\nRead: IV/RV > 1.3 = options RICH (the seller's edge — fear overpriced).")
    print("      IV/RV < 0.9 = options CHEAP (don't sell these — these are the BUY candidates).")
    print("      Only sell strikes you'd happily own at breakeven. OI < 100 = thin, use limits at mid.")
else:
    print("no candidates matched")


In [ ]:
# WEEK-VOL PRICER — what does SPY turbulence cost this week? Complete cell, run as-is.
import yfinance as yf
from datetime import date

t = yf.Ticker("SPY")
spot = t.history(period="1d")["Close"].iloc[-1]
print(f"SPY spot ${spot:.2f}  |  Week: CPI+Warsh Tue, Senate+PPI+BeigeBook Wed, TSMC Thu, opex/retail Fri\n")

for exp in t.options[:6]:
    dte = (date.fromisoformat(exp) - date.today()).days
    if dte < 1 or dte > 8: continue
    ch = t.option_chain(exp)
    atm_c = ch.calls.iloc[(ch.calls.strike - spot).abs().argsort()].iloc[0]
    atm_p = ch.puts.iloc[(ch.puts.strike - spot).abs().argsort()].iloc[0]
    strad = (atm_c.bid + atm_c.ask)/2 + (atm_p.bid + atm_p.ask)/2
    print(f"{exp} ({dte}d): ATM straddle ~${strad:.2f} = ±{strad/spot*100:.2f}% implied move "
          f"| call IV {atm_c.impliedVolatility*100:.1f} put IV {atm_p.impliedVolatility*100:.1f}")
print("\nRead: CPI days that MISS consensus have printed 1-2%+ moves this year (Mar CPI day, AVGO day).")
print("Straddle under ~0.8% into Tue = turbulence still underpriced (the corr discount suppressing it).")
print("Straddle 1.3%+ = the event is fully priced — adding long vol here just donates the IV crush.")
